# 웨이퍼 결함 — 올인원 파이프라인 (학습 + Graph-RAG + Streamlit)

한 노트북으로 전부 실행합니다.
1. **데이터/EDA** — MixedWM38 (`arr_0` 0/1/2, `arr_1` 8-멀티핫)
2. **객체 탐지 (YOLOv8n)** — 단일패턴 bbox (F-02 데모)
3. **다중라벨 분류 (Swin)** — 혼합 패턴 8종 동시 예측 (F-04 근거, 논문 핵심 macro-F1)
4. **Graph-RAG** — 공정 지식그래프 기반 원인 역추적 리포트 + groundedness 평가
5. **Streamlit 데모** — F-01~F-07 화면(업로드·탐지·3D주파수·리포트·피드백·PDF), Colab에서 바로 실행

> Colab: 런타임 → **GPU(T4)** 권장. 경로만 본인 환경에 맞게 수정하세요.

## 0. 설치

In [ ]:
!pip -q install ultralytics timm torch torchvision scikit-learn matplotlib scipy networkx reportlab streamlit transformers accelerate
print('done')

## 1. 데이터 로드 + EDA

In [ ]:
import numpy as np, os, json, glob
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

NPZ_PATH = 'MixedWM38.npz'   # 예: '/content/drive/MyDrive/superb-spot/MixedWM38.npz'
data = np.load(NPZ_PATH)
X = data['arr_0']; Y = data['arr_1'].astype(int)
LABELS = ['Center','Donut','Edge_Loc','Edge_Ring','Loc','Near_Full','Scratch','Random']
json.dump({'labels':LABELS}, open('label_map.json','w'), ensure_ascii=False, indent=2)
print('X', X.shape, 'Y', Y.shape)
for k,l in enumerate(LABELS): print(f'{l:10s}: {int(Y[:,k].sum())}')
print('정상', int((Y.sum(1)==0).sum()), '| 단일', int((Y.sum(1)==1).sum()), '| 혼합', int((Y.sum(1)>=2).sum()))

cmap = ListedColormap(['#0b1021','#94a3b8','#ef4444'])
idxs = np.random.RandomState(0).choice(len(X), 8, replace=False)
plt.figure(figsize=(14,4))
for j,i in enumerate(idxs):
    plt.subplot(2,4,j+1); plt.imshow(X[i], cmap=cmap, vmin=0, vmax=2)
    act=[LABELS[k] for k in range(8) if Y[i,k]==1] or ['Normal']
    plt.title('+'.join(act), fontsize=8); plt.axis('off')
plt.tight_layout(); plt.show()

## 2. 객체 탐지 (YOLOv8n) — 단일패턴 bbox (F-02)
단일 패턴 이미지의 결함 픽셀 바운딩박스를 자동 생성 → YOLOv8n 파인튜닝. (경계 inset·크기 상한으로 발산 방지)

In [ ]:
from scipy import ndimage
from PIL import Image
from sklearn.model_selection import train_test_split
from pathlib import Path

IMG=256; S=IMG/52.0; M=3.0; MAXF=0.85; MINS=10.0
root=Path('wafer_yolo')
for s in ['images/train','images/val','labels/train','labels/val']:(root/s).mkdir(parents=True,exist_ok=True)

def to_rgb(a):
    rgb=np.zeros((*a.shape,3),np.uint8); rgb[a==1]=(148,163,184); rgb[a==2]=(239,68,68)
    return Image.fromarray(rgb).resize((IMG,IMG),Image.NEAREST)

def clamp(bx0,by0,bx1,by1):
    cx,cy=(bx0+bx1)/2,(by0+by1)/2
    w=min(max(bx1-bx0,MINS),MAXF*IMG); h=min(max(by1-by0,MINS),MAXF*IMG)
    bx0,bx1,by0,by1=cx-w/2,cx+w/2,cy-h/2,cy+h/2
    if bx0<M: bx1+=M-bx0; bx0=M
    if by0<M: by1+=M-by0; by0=M
    if bx1>IMG-M: bx0-=bx1-(IMG-M); bx1=IMG-M
    if by1>IMG-M: by0-=by1-(IMG-M); by1=IMG-M
    return max(M,bx0),max(M,by0),min(IMG-M,bx1),min(IMG-M,by1)

def yolo_box(a,cls):
    mask=(a==2)
    if mask.sum()==0: return None
    lbl,n=ndimage.label(mask); comps=[]
    for c in range(1,n+1):
        ys,xs=np.where(lbl==c)
        if len(xs)>=3: comps.append((xs.min(),ys.min(),xs.max(),ys.max(),len(xs)))
    if not comps: return None
    name=LABELS[cls]
    if name in ('Near_Full','Random'):
        ys,xs=np.where(mask); x0,y0,x1,y1=xs.min(),ys.min(),xs.max(),ys.max()
    else:
        x0,y0,x1,y1,_=max(comps,key=lambda c:c[4])
    bx0,by0,bx1,by1=clamp(x0*S,y0*S,(x1+1)*S,(y1+1)*S)
    cx=((bx0+bx1)/2)/IMG; cy=((by0+by1)/2)/IMG; w=(bx1-bx0)/IMG; h=(by1-by0)/IMG
    return (cls,cx,cy,w,h)

single=np.where(Y.sum(1)==1)[0]
cls_of={i:int(np.argmax(Y[i])) for i in single}
tr,va=train_test_split(single,test_size=0.2,random_state=42,stratify=[cls_of[i] for i in single])
def dump(split,ids):
    k=0
    for i in ids:
        b=yolo_box(X[i],cls_of[i])
        if not b: continue
        to_rgb(X[i]).save(root/f'images/{split}/wafer_{i}.png')
        with open(root/f'labels/{split}/wafer_{i}.txt','w') as f:
            f.write('%d %.6f %.6f %.6f %.6f\n'%b)
        k+=1
    return k
print('train',dump('train',tr),'val',dump('val',va))
open(root/'data.yaml','w').write(f"path: {root.resolve()}\ntrain: images/train\nval: images/val\nnc: 8\nnames: {LABELS}\n")

In [ ]:
from ultralytics import YOLO
det=YOLO('yolov8n.pt')
det.train(data=str(root/'data.yaml'), epochs=40, imgsz=IMG, batch=32, patience=10, name='wafer_det')
m=det.val(); print('mAP50',m.box.map50,'mAP50-95',m.box.map)

## 3. 다중라벨 분류 (Swin) — 논문 핵심 (macro-F1 · 혼동행렬)
혼합 패턴까지 8종을 동시에 예측. `arr_1`을 타깃으로 사용.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import f1_score, multilabel_confusion_matrix, classification_report

DEV='cuda' if torch.cuda.is_available() else 'cpu'; C=224
def rgb(a):
    r=np.zeros((*a.shape,3),np.uint8); r[a==1]=(148,163,184); r[a==2]=(239,68,68)
    return np.array(Image.fromarray(r).resize((C,C),Image.NEAREST))
class DS(Dataset):
    def __init__(s,idx): s.idx=idx
    def __len__(s): return len(s.idx)
    def __getitem__(s,j):
        i=s.idx[j]; im=rgb(X[i]).astype(np.float32)/255.; im=(im-0.5)/0.5
        return torch.tensor(im).permute(2,0,1), torch.tensor(Y[i],dtype=torch.float32)
tr_i,va_i=train_test_split(np.arange(len(X)),test_size=0.15,random_state=7)
tl=DataLoader(DS(tr_i),batch_size=64,shuffle=True,num_workers=2)
vl=DataLoader(DS(va_i),batch_size=128,num_workers=2)
net=timm.create_model('swin_tiny_patch4_window7_224',pretrained=True,num_classes=8).to(DEV)
opt=torch.optim.AdamW(net.parameters(),lr=1e-4,weight_decay=0.05); crit=nn.BCEWithLogitsLoss()
for ep in range(8):
    net.train()
    for xb,yb in tl:
        opt.zero_grad(); loss=crit(net(xb.to(DEV)),yb.to(DEV)); loss.backward(); opt.step()
    net.eval(); P=[];T=[]
    with torch.no_grad():
        for xb,yb in vl:
            P.append(torch.sigmoid(net(xb.to(DEV))).cpu().numpy()); T.append(yb.numpy())
    P=np.concatenate(P); T=np.concatenate(T)
    f1=f1_score(T,(P>0.5).astype(int),average='macro',zero_division=0)
    print(f'epoch {ep+1}/8  macro-F1={f1:.4f}')
torch.save(net.state_dict(),'swin_multilabel.pt')
print('\n=== 최종 리포트 ===')
print(classification_report(T,(P>0.5).astype(int),target_names=LABELS,zero_division=0))

In [ ]:
# 패턴별 F1 막대 + 혼동행렬(멀티라벨)
per=f1_score(T,(P>0.5).astype(int),average=None,zero_division=0)
plt.figure(figsize=(9,3)); plt.bar(LABELS,per,color='#60a5fa'); plt.ylim(0,1)
plt.title('패턴별 F1'); plt.xticks(rotation=30); plt.tight_layout(); plt.show()
mcm=multilabel_confusion_matrix(T,(P>0.5).astype(int))
fig,ax=plt.subplots(2,4,figsize=(14,6))
for k,a in enumerate(ax.flat):
    a.imshow(mcm[k],cmap='Blues'); a.set_title(LABELS[k],fontsize=9)
    for (r,c),v in np.ndenumerate(mcm[k]): a.text(c,r,int(v),ha='center',va='center')
    a.set_xticks([0,1]); a.set_yticks([0,1])
plt.suptitle('클래스별 혼동행렬'); plt.tight_layout(); plt.show()

## 4. Graph-RAG — 공정 지식그래프 원인 역추적 (F-04)

In [ ]:
"""
Graph-RAG 원인 역추적 (기획서 F-04)

공정 지식그래프(Process KG)에서 예측 패턴 → 유발 공정 → 원인 → 설비 → 근거를
검색해 '설명 가능한' 리포트를 생성. LLM 없이도 구조적 리포트로 동작하며,
USE_LLM=True + transformers 설치 시 로컬 LLM(Qwen2.5)으로 문장화.
"""
import networkx as nx

TRIPLES = [
    ('Center','causes_process','CMP_Polishing'), ('CMP_Polishing','via_cause','Uneven_polish_pressure_slurry'), ('CMP_Polishing','on_equip','CMP_tool'),
    ('Donut','causes_process','Deposition_Litho'), ('Deposition_Litho','via_cause','Coating_thickness_variation'), ('Deposition_Litho','on_equip','Coater_Scanner'),
    ('Edge_Loc','causes_process','Etch'), ('Etch','via_cause','Edge_plasma_nonuniformity'), ('Etch','on_equip','Plasma_etcher'),
    ('Edge_Ring','causes_process','Anneal_Etch'), ('Anneal_Etch','via_cause','Edge_temperature_etch_nonuniformity'), ('Anneal_Etch','on_equip','Anneal_furnace'),
    ('Loc','causes_process','Lithography'), ('Lithography','via_cause','Focus_deviation'), ('Lithography','on_equip','Scanner_Stepper'),
    ('Near_Full','causes_process','Wide_process_fault'), ('Wide_process_fault','via_cause','Equipment_recipe_major_fault'), ('Wide_process_fault','on_equip','Fab_line'),
    ('Scratch','causes_process','Handling_Dicing'), ('Handling_Dicing','via_cause','Physical_scratch_handling'), ('Handling_Dicing','on_equip','Wafer_robot_arm'),
    ('Random','causes_process','Cleaning'), ('Cleaning','via_cause','Cleanroom_particle_contamination'), ('Cleaning','on_equip','Cleaning_bath'),
    ('Center','evidence','GlobalSino_pattern_cause'), ('Edge_Ring','evidence','Wavelet2023_frequency'),
    ('Scratch','evidence','GlobalSino_pattern_cause'), ('Center','evidence','Wang2020_MixedWM38'),
]

HUMAN = {
    'CMP_Polishing':'CMP(화학기계연마)','Deposition_Litho':'증착/노광','Etch':'식각','Anneal_Etch':'어닐링/식각',
    'Lithography':'노광','Wide_process_fault':'광범위 공정 이상','Handling_Dicing':'이송/다이싱','Cleaning':'세정',
    'Uneven_polish_pressure_slurry':'연마 압력/슬러리 불균일','Coating_thickness_variation':'도포 두께 편차',
    'Edge_plasma_nonuniformity':'가장자리 플라즈마 분포 불균일','Edge_temperature_etch_nonuniformity':'가장자리 온도/식각 불균일',
    'Focus_deviation':'초점 이탈','Equipment_recipe_major_fault':'설비/레시피 대규모 이상',
    'Physical_scratch_handling':'핸들링 중 물리적 긁힘','Cleanroom_particle_contamination':'클린룸 입자 오염',
}
# 공정 단계 번호(리포트 문구용)
PROC_STEP = {
    'Lithography':'1단계: 노광', 'Etch':'2단계: 식각', 'CMP_Polishing':'3단계: 화학 기계적 연마',
    'Deposition_Litho':'증착/노광 공정', 'Anneal_Etch':'어닐링/식각 공정',
    'Wide_process_fault':'광범위 공정', 'Handling_Dicing':'이송/다이싱 공정', 'Cleaning':'세정 공정',
}
# F-05 드롭다운용 공정 목록
PROCESS_CHOICES = ['자동 인식','노광(Lithography)','식각(Etch)','화학기계연마(CMP)','증착(Deposition)',
                   '어닐링(Anneal)','세정(Cleaning)','이송/다이싱(Handling/Dicing)']

def _h(x): return HUMAN.get(x, x)

def build_graph():
    G = nx.MultiDiGraph()
    for s, r, o in TRIPLES:
        G.add_edge(s, o, rel=r)
    return G

G = build_graph()

def retrieve(patterns):
    facts = []
    for p in patterns:
        for _, o, d in G.out_edges(p, data=True):
            if d['rel'] == 'causes_process':
                proc = o
                facts.append((p, '유발 공정', _h(proc)))
                for _, o2, d2 in G.out_edges(proc, data=True):
                    if d2['rel'] == 'via_cause': facts.append((_h(proc), '원인', _h(o2)))
                    if d2['rel'] == 'on_equip': facts.append((_h(proc), '설비', o2))
            if d['rel'] == 'evidence':
                facts.append((p, '근거', o))
    return facts

def primary_process(patterns):
    """가장 대표적인 유발 공정 문구(신뢰도 표기용)."""
    for p in patterns:
        for _, o, d in G.out_edges(p, data=True):
            if d['rel'] == 'causes_process':
                return PROC_STEP.get(o, _h(o))
    return '미상 공정'

def report(patterns, confidence=None, use_llm=False):
    """설명 가능한 원인 역추적 리포트(문자열)."""
    if not patterns:
        return '결함 패턴이 탐지되지 않았습니다. (정상 웨이퍼로 판단)'
    if use_llm:
        try:
            out = _llm_report(patterns)
            if out:
                return out
        except Exception:
            pass
    lines = []
    proc = primary_process(patterns)
    conf = f' (신뢰도 {confidence:.0%})' if confidence is not None else ''
    lines.append(f'현재 이미지는 [{proc}]에서 발생한 불량으로 판단됨{conf}.')
    lines.append('')
    for p in patterns:
        procs = [o for _, o, d in G.out_edges(p, data=True) if d['rel'] == 'causes_process']
        evs = [o for _, o, d in G.out_edges(p, data=True) if d['rel'] == 'evidence']
        for pr in procs:
            causes = [_h(o2) for _, o2, d2 in G.out_edges(pr, data=True) if d2['rel'] == 'via_cause']
            equips = [o2 for _, o2, d2 in G.out_edges(pr, data=True) if d2['rel'] == 'on_equip']
            cite = ''.join(f'[{e}]' for e in evs)
            lines.append(f'· {p}: {_h(pr)} 공정의 {", ".join(causes)}(으)로 판단. '
                         f'관련 설비: {", ".join(equips)} {cite}')
    return '\n'.join(lines)

def _llm_report(patterns):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if not torch.cuda.is_available():
        return None
    global _TOK, _LM
    try:
        _TOK
    except NameError:
        name = 'Qwen/Qwen2.5-3B-Instruct'
        _TOK = AutoTokenizer.from_pretrained(name)
        _LM = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map='auto')
    ctx = '\n'.join('- %s | %s | %s' % f for f in retrieve(patterns))
    prompt = ('너는 반도체 공정 엔지니어다. 아래 지식그래프 사실만 사용해 3~5문장 원인 리포트를 쓰고, '
              '근거는 [키] 형태로 인용하라. 사실에 없는 내용은 지어내지 마라.\n'
              f'탐지 패턴: {", ".join(patterns)}\n지식그래프 사실:\n{ctx}')
    msgs = [{'role': 'user', 'content': prompt}]
    text = _TOK.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = _TOK(text, return_tensors='pt').to(_LM.device)
    out = _LM.generate(**inp, max_new_tokens=300, do_sample=False)
    return _TOK.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


In [ ]:
# 리포트 예시 + groundedness 평가(룰 vs 순수LLM vs Graph-RAG)
def kg_terms(ps): return {o for _,_,o in retrieve(ps) if isinstance(o,str)}
def grounded(rep,ps):
    t=kg_terms(ps); return sum(1 for x in t if x in rep)/max(len(t),1)
for ps in [['Center'],['Edge_Ring','Scratch'],['Donut','Loc','Random']]:
    r=report(ps,use_llm=False)
    print('패턴',ps,'→ groundedness %.2f'%grounded(r,ps))
    print(r,'\n')

## 5. Streamlit 데모 (F-01~F-07)
아래 셀들이 앱 파일을 디스크에 쓰고, Colab에서 실행합니다. (탐지=best.pt, 분류=swin_multilabel.pt 자동 로드; 없으면 휴리스틱 폴백)

In [ ]:
%%writefile superb_client.py
"""
Superb AI 연동 모듈
  F-01 업로드 / F-02 배포 모델 추론(오토라벨링) / F-05 피드백 루프

- API 키는 절대 코드에 넣지 말 것. 환경변수 SUPERB_AI_API_KEY 로만 주입.
    export SUPERB_AI_API_KEY=sbd_pk_...
- 키가 없거나 SDK 미설치면 모든 함수가 조용히 실패(ok=False)해서
  데모가 오프라인으로도 계속 동작합니다.
"""
from __future__ import annotations
import os
from typing import Optional

TENANT = os.environ.get("SUPERB_TENANT", "spot")
PROJECT_ID = os.environ.get("SUPERB_PROJECT_ID", "7b267a4e-eea0-4764-9a61-6d74515a4985")
DATASET_ID = os.environ.get("SUPERB_DATASET_ID", "ded47876-a675-492e-a548-c296f80fd151")
CLS_ID = os.environ.get("SUPERB_CLASS_ID", "9fd96ef0-0b00-4157-8633-fabd20a4e6c3")
# 배포한 탐지 모델 (RF-DETR). 모델 페이지 하단의 "모델 ID" 또는 배포 ID.
MODEL_ID = os.environ.get("SUPERB_MODEL_ID", "")
DEPLOY_ID = os.environ.get("SUPERB_DEPLOYMENT_ID", "")

LABELS = ["Center", "Donut", "Edge_Loc", "Edge_Ring", "Loc", "Near_Full", "Scratch", "Random"]


def _client():
    if not os.environ.get("SUPERB_AI_API_KEY"):
        return None
    try:
        from superb_ai import Client
    except Exception:
        return None
    try:
        return Client(tenant=TENANT)
    except Exception:
        return None


def available() -> bool:
    return _client() is not None


def deploy_available() -> bool:
    """배포 모델 추론이 가능한 상태인지."""
    return _client() is not None and bool(MODEL_ID or DEPLOY_ID)


# ---------------- F-01 업로드 ----------------
def upload_image(path: str, key: Optional[str] = None) -> dict:
    c = _client()
    if c is None:
        return {"ok": False, "reason": "no_api_key_or_sdk"}
    try:
        res = c.assets.upload_paths(DATASET_ID, [path], concurrency=1)
        r0 = res[0] if isinstance(res, list) and res else res
        aid = str(getattr(r0, "asset_id", getattr(r0, "id", "")))
        # 업로드된 자산을 프로젝트 범위에 편입해야 어노테이션이 가능
        if aid:
            try:
                c.project_assets.batch_add(PROJECT_ID, asset_ids=[aid])
            except Exception:
                pass
        return {"ok": True, "asset_id": aid, "key": key or os.path.basename(path)}
    except Exception as e:
        return {"ok": False, "reason": str(e)}


# ---------------- F-02 배포 모델 추론 ----------------
def detect_remote(image_path: str, conf: float = 0.25) -> dict:
    """
    배포된 Superb 탐지 모델로 추론.
    반환: {'ok':True, 'boxes':[(label,x0,y0,x1,y1,conf), ...]} 또는 {'ok':False,'reason':...}
    SDK 버전마다 시그니처가 달라 여러 형태를 순차 시도합니다.
    """
    c = _client()
    if c is None:
        return {"ok": False, "reason": "no_api_key_or_sdk"}
    if not (MODEL_ID or DEPLOY_ID):
        return {"ok": False, "reason": "no_model_id (SUPERB_MODEL_ID 환경변수를 설정하세요)"}

    mid = DEPLOY_ID or MODEL_ID
    attempts = []
    for res_name in ("deployments", "models", "foundation"):
        res = getattr(c, res_name, None)
        if res is None:
            continue
        for meth in ("predict", "infer", "inference", "run", "predict_file", "predict_image"):
            fn = getattr(res, meth, None)
            if not callable(fn):
                continue
            for kwargs in (
                {"deployment_id": mid, "file_path": image_path},
                {"model_id": mid, "file_path": image_path},
                {"deployment_id": mid, "path": image_path},
                {"model_id": mid, "path": image_path},
                {"deployment_id": mid, "image": image_path},
                {"model_id": mid, "image": image_path},
            ):
                try:
                    out = fn(**kwargs)
                    return {"ok": True, "boxes": _parse_boxes(out, conf), "via": f"{res_name}.{meth}"}
                except TypeError:
                    continue
                except Exception as e:
                    attempts.append(f"{res_name}.{meth}: {type(e).__name__}")
                    break
    return {"ok": False, "reason": "no_matching_predict_api", "tried": attempts[:6]}


def _parse_boxes(out, conf_thr: float):
    """SDK 응답에서 (label, x0,y0,x1,y1, conf) 리스트 추출 (형태 방어적 처리)."""
    def as_dict(o):
        if isinstance(o, dict):
            return o
        if hasattr(o, "model_dump"):
            return o.model_dump()
        try:
            return dict(o)
        except Exception:
            return getattr(o, "__dict__", {}) or {}

    d = as_dict(out)
    items = None
    for key in ("predictions", "annotations", "objects", "results", "boxes", "items"):
        if isinstance(d.get(key), list):
            items = d[key]
            break
    if items is None and isinstance(out, list):
        items = out
    if items is None:
        return []

    boxes = []
    for it in items:
        e = as_dict(it)
        score = float(e.get("score", e.get("confidence", e.get("conf", 1.0))) or 0.0)
        if score < conf_thr:
            continue
        name = e.get("class_name") or e.get("label") or e.get("class") or e.get("name") or "Defect"
        g = e.get("geometry") or e.get("bbox") or e.get("box") or e
        g = as_dict(g)
        if all(k in g for k in ("x", "y", "w", "h")):
            x, y, w, h = float(g["x"]), float(g["y"]), float(g["w"]), float(g["h"])
            boxes.append((str(name), x, y, x + w, y + h, score))
        elif all(k in g for k in ("x1", "y1", "x2", "y2")):
            boxes.append((str(name), float(g["x1"]), float(g["y1"]), float(g["x2"]), float(g["y2"]), score))
        elif isinstance(g, (list, tuple)) and len(g) == 4:
            x0, y0, x1, y1 = map(float, g)
            boxes.append((str(name), x0, y0, x1, y1, score))
    return boxes


# ---------------- 어노테이션 주입 / F-05 피드백 ----------------
def push_prediction(asset_id: str, patterns: list) -> dict:
    c = _client()
    if c is None:
        return {"ok": False, "reason": "no_api_key_or_sdk"}
    answer = [p for p in patterns if p in LABELS]
    if not answer:
        return {"ok": False, "reason": "empty_prediction"}
    ann = [{"asset_id": str(asset_id), "class_id": CLS_ID,
            "type": "classification", "data": {"answer": answer}}]
    try:
        r = c.annotations.project_batch_create(project_id=PROJECT_ID, annotations=ann, replace=True)
        return {"ok": True, "result": str(r)}
    except Exception as e:
        return {"ok": False, "reason": str(e)}


def push_feedback(asset_id: str, corrected_patterns: list) -> dict:
    """F-05: 엔지니어 수정 라벨을 Superb로 재주입 (액티브 러닝)."""
    return push_prediction(asset_id, corrected_patterns)


In [ ]:
%%writefile graph_rag.py
"""
Graph-RAG 원인 역추적 (기획서 F-04)

공정 지식그래프(Process KG)에서 예측 패턴 → 유발 공정 → 원인 → 설비 → 근거를
검색해 '설명 가능한' 리포트를 생성. LLM 없이도 구조적 리포트로 동작하며,
USE_LLM=True + transformers 설치 시 로컬 LLM(Qwen2.5)으로 문장화.
"""
from __future__ import annotations
import networkx as nx

TRIPLES = [
    ('Center','causes_process','CMP_Polishing'), ('CMP_Polishing','via_cause','Uneven_polish_pressure_slurry'), ('CMP_Polishing','on_equip','CMP_tool'),
    ('Donut','causes_process','Deposition_Litho'), ('Deposition_Litho','via_cause','Coating_thickness_variation'), ('Deposition_Litho','on_equip','Coater_Scanner'),
    ('Edge_Loc','causes_process','Etch'), ('Etch','via_cause','Edge_plasma_nonuniformity'), ('Etch','on_equip','Plasma_etcher'),
    ('Edge_Ring','causes_process','Anneal_Etch'), ('Anneal_Etch','via_cause','Edge_temperature_etch_nonuniformity'), ('Anneal_Etch','on_equip','Anneal_furnace'),
    ('Loc','causes_process','Lithography'), ('Lithography','via_cause','Focus_deviation'), ('Lithography','on_equip','Scanner_Stepper'),
    ('Near_Full','causes_process','Wide_process_fault'), ('Wide_process_fault','via_cause','Equipment_recipe_major_fault'), ('Wide_process_fault','on_equip','Fab_line'),
    ('Scratch','causes_process','Handling_Dicing'), ('Handling_Dicing','via_cause','Physical_scratch_handling'), ('Handling_Dicing','on_equip','Wafer_robot_arm'),
    ('Random','causes_process','Cleaning'), ('Cleaning','via_cause','Cleanroom_particle_contamination'), ('Cleaning','on_equip','Cleaning_bath'),
    ('Center','evidence','GlobalSino_pattern_cause'), ('Edge_Ring','evidence','Wavelet2023_frequency'),
    ('Scratch','evidence','GlobalSino_pattern_cause'), ('Center','evidence','Wang2020_MixedWM38'),
]

HUMAN = {
    'CMP_Polishing':'CMP(화학기계연마)','Deposition_Litho':'증착/노광','Etch':'식각','Anneal_Etch':'어닐링/식각',
    'Lithography':'노광','Wide_process_fault':'광범위 공정 이상','Handling_Dicing':'이송/다이싱','Cleaning':'세정',
    'Uneven_polish_pressure_slurry':'연마 압력/슬러리 불균일','Coating_thickness_variation':'도포 두께 편차',
    'Edge_plasma_nonuniformity':'가장자리 플라즈마 분포 불균일','Edge_temperature_etch_nonuniformity':'가장자리 온도/식각 불균일',
    'Focus_deviation':'초점 이탈','Equipment_recipe_major_fault':'설비/레시피 대규모 이상',
    'Physical_scratch_handling':'핸들링 중 물리적 긁힘','Cleanroom_particle_contamination':'클린룸 입자 오염',
}
# 공정 단계 번호(리포트 문구용)
PROC_STEP = {
    'Lithography':'1단계: 노광', 'Etch':'2단계: 식각', 'CMP_Polishing':'3단계: 화학 기계적 연마',
    'Deposition_Litho':'증착/노광 공정', 'Anneal_Etch':'어닐링/식각 공정',
    'Wide_process_fault':'광범위 공정', 'Handling_Dicing':'이송/다이싱 공정', 'Cleaning':'세정 공정',
}
# F-05 드롭다운용 공정 목록
PROCESS_CHOICES = ['자동 인식','노광(Lithography)','식각(Etch)','화학기계연마(CMP)','증착(Deposition)',
                   '어닐링(Anneal)','세정(Cleaning)','이송/다이싱(Handling/Dicing)']

def _h(x): return HUMAN.get(x, x)

def build_graph():
    G = nx.MultiDiGraph()
    for s, r, o in TRIPLES:
        G.add_edge(s, o, rel=r)
    return G

G = build_graph()

def retrieve(patterns):
    facts = []
    for p in patterns:
        for _, o, d in G.out_edges(p, data=True):
            if d['rel'] == 'causes_process':
                proc = o
                facts.append((p, '유발 공정', _h(proc)))
                for _, o2, d2 in G.out_edges(proc, data=True):
                    if d2['rel'] == 'via_cause': facts.append((_h(proc), '원인', _h(o2)))
                    if d2['rel'] == 'on_equip': facts.append((_h(proc), '설비', o2))
            if d['rel'] == 'evidence':
                facts.append((p, '근거', o))
    return facts

def primary_process(patterns):
    """가장 대표적인 유발 공정 문구(신뢰도 표기용)."""
    for p in patterns:
        for _, o, d in G.out_edges(p, data=True):
            if d['rel'] == 'causes_process':
                return PROC_STEP.get(o, _h(o))
    return '미상 공정'

def report(patterns, confidence=None, use_llm=False):
    """설명 가능한 원인 역추적 리포트(문자열)."""
    if not patterns:
        return '결함 패턴이 탐지되지 않았습니다. (정상 웨이퍼로 판단)'
    if use_llm:
        try:
            out = _llm_report(patterns)
            if out:
                return out
        except Exception:
            pass
    lines = []
    proc = primary_process(patterns)
    conf = f' (신뢰도 {confidence:.0%})' if confidence is not None else ''
    lines.append(f'현재 이미지는 [{proc}]에서 발생한 불량으로 판단됨{conf}.')
    lines.append('')
    for p in patterns:
        procs = [o for _, o, d in G.out_edges(p, data=True) if d['rel'] == 'causes_process']
        evs = [o for _, o, d in G.out_edges(p, data=True) if d['rel'] == 'evidence']
        for pr in procs:
            causes = [_h(o2) for _, o2, d2 in G.out_edges(pr, data=True) if d2['rel'] == 'via_cause']
            equips = [o2 for _, o2, d2 in G.out_edges(pr, data=True) if d2['rel'] == 'on_equip']
            cite = ''.join(f'[{e}]' for e in evs)
            lines.append(f'· {p}: {_h(pr)} 공정의 {", ".join(causes)}(으)로 판단. '
                         f'관련 설비: {", ".join(equips)} {cite}')
    return '\n'.join(lines)

def _llm_report(patterns):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if not torch.cuda.is_available():
        return None
    global _TOK, _LM
    try:
        _TOK
    except NameError:
        name = 'Qwen/Qwen2.5-3B-Instruct'
        _TOK = AutoTokenizer.from_pretrained(name)
        _LM = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map='auto')
    ctx = '\n'.join('- %s | %s | %s' % f for f in retrieve(patterns))
    prompt = ('너는 반도체 공정 엔지니어다. 아래 지식그래프 사실만 사용해 3~5문장 원인 리포트를 쓰고, '
              '근거는 [키] 형태로 인용하라. 사실에 없는 내용은 지어내지 마라.\n'
              f'탐지 패턴: {", ".join(patterns)}\n지식그래프 사실:\n{ctx}')
    msgs = [{'role': 'user', 'content': prompt}]
    text = _TOK.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = _TOK(text, return_tensors='pt').to(_LM.device)
    out = _LM.generate(**inp, max_new_tokens=300, do_sample=False)
    return _TOK.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


In [ ]:
%%writefile models.py
"""
모델 로딩 & 추론
  탐지: Superb 배포 모델(RF-DETR) → 로컬 YOLO(best.pt) → 휴리스틱 폴백
  분류: Swin(swin_multilabel.pt) → 휴리스틱 폴백
학습 결과 파일(best.pt / swin_multilabel.pt)을 같은 폴더에 두면 자동 사용.
"""
from __future__ import annotations
import os
import tempfile
import numpy as np
from PIL import Image

LABELS = ["Center", "Donut", "Edge_Loc", "Edge_Ring", "Loc", "Near_Full", "Scratch", "Random"]
DET_PATH = os.environ.get("WAFER_DET_PT", "best.pt")
CLS_PATH = os.environ.get("WAFER_CLS_PT", "swin_multilabel.pt")


# ---------- 입력 전처리 ----------
def image_to_mask(img: Image.Image, size: int = 52) -> np.ndarray:
    """웨이퍼 PNG → 52x52 마스크(0 빈, 1 정상, 2 불량)."""
    a = np.array(img.convert("RGB").resize((size, size), Image.NEAREST)).astype(int)
    r, g, b = a[..., 0], a[..., 1], a[..., 2]
    defect = (r > 150) & (g < 130) & (b < 130)
    normal = (~defect) & (a.sum(-1) > 120)
    m = np.zeros((size, size), int)
    m[normal] = 1
    m[defect] = 2
    return m


# ---------- 분류 (다중라벨) ----------
_swin = None
def _load_swin():
    global _swin
    if _swin is not None:
        return _swin
    if not os.path.exists(CLS_PATH):
        return None
    try:
        import torch, timm
        net = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=8)
        net.load_state_dict(torch.load(CLS_PATH, map_location="cpu"))
        net.eval()
        _swin = net
        return net
    except Exception:
        return None


def classify(img: Image.Image, thr: float = 0.5):
    """반환: (labels, scores, source)"""
    net = _load_swin()
    if net is not None:
        import torch
        arr = np.array(img.convert("RGB").resize((224, 224), Image.NEAREST)).astype(np.float32) / 255.0
        arr = (arr - 0.5) / 0.5
        x = torch.tensor(arr).permute(2, 0, 1).unsqueeze(0)
        with torch.no_grad():
            p = torch.sigmoid(net(x)).numpy()[0]
        scores = {LABELS[k]: float(p[k]) for k in range(8)}
        preds = [LABELS[k] for k in range(8) if p[k] > thr]
        return preds, scores, "Swin 다중라벨(학습모델)"
    return _heuristic_classify(image_to_mask(img))


def _heuristic_classify(m: np.ndarray):
    H = W = m.shape[0]
    defect = (m == 2)
    total = defect.sum()
    scores = {l: 0.0 for l in LABELS}
    if total < 5:
        return [], scores, "휴리스틱(정상)"
    ys, xs = np.where(defect)
    yy, xx = np.mgrid[0:H, 0:W]
    rr = np.sqrt((yy - H / 2) ** 2 + (xx - W / 2) ** 2) / (H / 2)
    edge_ratio = defect[rr > 0.75].sum() / max(total, 1)
    center_ratio = defect[rr < 0.35].sum() / max(total, 1)
    density = total / (np.pi * (H / 2) ** 2)
    if total >= 8:
        c = np.cov(np.stack([xs, ys]))
        ev = np.sort(np.linalg.eigvalsh(c))
        elong = ev[1] / max(ev[0], 1e-6)
    else:
        elong = 1.0
    scores["Near_Full"] = min(density * 2.5, 1.0)
    scores["Edge_Ring"] = edge_ratio
    scores["Edge_Loc"] = edge_ratio * 0.7
    scores["Center"] = center_ratio
    scores["Loc"] = center_ratio * 0.6
    scores["Scratch"] = min(elong / 12.0, 1.0)
    scores["Random"] = min(density * 1.2, 1.0) if elong < 3 and edge_ratio < 0.4 and center_ratio < 0.4 else 0.2
    preds = [l for l, s in scores.items() if s > 0.5] or [max(scores, key=scores.get)]
    return preds, scores, "휴리스틱(데모)"


# ---------- 탐지 ----------
_yolo = None
def _load_yolo():
    global _yolo
    if _yolo is not None:
        return _yolo
    if not os.path.exists(DET_PATH):
        return None
    try:
        from ultralytics import YOLO
        _yolo = YOLO(DET_PATH)
        return _yolo
    except Exception:
        return None


def yolo_available() -> bool:
    return os.path.exists(DET_PATH)


def detect(img: Image.Image, source: str = "auto", conf: float = 0.25):
    """
    source: 'auto' | 'superb' | 'yolo' | 'heuristic'
    반환: (boxes[(label,x0,y0,x1,y1,conf)], source_str)
    """
    # 1) Superb 배포 모델 (RF-DETR)
    if source in ("auto", "superb"):
        try:
            import superb_client as sb
            if sb.deploy_available():
                with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
                    tmp = f.name
                img.convert("RGB").save(tmp)
                r = sb.detect_remote(tmp, conf=conf)
                os.unlink(tmp)
                if r.get("ok") and r.get("boxes"):
                    return _scale_boxes(r["boxes"], img), "Superb RF-DETR(배포모델)"
                if source == "superb":
                    return [], f"Superb 호출 실패({r.get('reason','?')})"
            elif source == "superb":
                return [], "Superb 배포모델 미설정 (SUPERB_MODEL_ID / API 키 확인)"
        except Exception as e:
            if source == "superb":
                return [], f"Superb 오류({e})"

    # 2) 로컬 YOLO
    if source in ("auto", "yolo"):
        model = _load_yolo()
        if model is not None:
            r = model.predict(np.array(img.convert("RGB")), verbose=False, conf=conf)[0]
            boxes = []
            for b in r.boxes:
                x0, y0, x1, y1 = b.xyxy[0].tolist()
                boxes.append((LABELS[int(b.cls[0])], x0, y0, x1, y1, float(b.conf[0])))
            return boxes, "YOLOv8(학습모델)"
        if source == "yolo":
            return [], "YOLO 가중치(best.pt) 없음"

    # 3) 휴리스틱
    return _heuristic_detect(img)


def _scale_boxes(boxes, img: Image.Image):
    """모델이 256px 기준으로 반환한 좌표를 현재 이미지 크기로 스케일."""
    W, H = img.size
    sx, sy = W / 256.0, H / 256.0
    if abs(sx - 1) < 1e-6 and abs(sy - 1) < 1e-6:
        return boxes
    return [(n, x0 * sx, y0 * sy, x1 * sx, y1 * sy, c) for (n, x0, y0, x1, y1, c) in boxes]


def _heuristic_detect(img: Image.Image):
    from scipy import ndimage
    W0, H0 = img.size
    m = image_to_mask(img, 52)
    mask = (m == 2)
    if mask.sum() == 0:
        return [], "휴리스틱(탐지없음)"
    lbl, n = ndimage.label(mask)
    sx, sy = W0 / 52.0, H0 / 52.0
    preds, _, _ = _heuristic_classify(m)
    name = preds[0] if preds else "Defect"
    boxes = []
    for c in range(1, n + 1):
        ys, xs = np.where(lbl == c)
        if len(xs) < 4:
            continue
        boxes.append((name, xs.min() * sx, ys.min() * sy,
                      (xs.max() + 1) * sx, (ys.max() + 1) * sy, 0.5))
    return boxes, "휴리스틱(데모)"


In [ ]:
%%writefile app.py
"""
웨이퍼 결함 분석 데모 — 기획서 F-01 ~ F-07 통합 (Streamlit)

실행:
    pip install streamlit torch timm ultralytics scipy reportlab pillow matplotlib networkx
    streamlit run app.py

기능 매핑
  F-01 이미지 업로드 + Superb 저장    F-02 결함 자동 탐지(빨간 박스)
  F-03 3D 주파수 신호 시각화         F-04 공정 판단 + 원인 역추적 리포트(Graph-RAG)
  F-05 공정 강제수정 + 피드백 루프     F-06 파일 예외처리
  F-07 결함 분석 리포트 PDF 다운로드
"""
import io
import os
import datetime
import numpy as np
import streamlit as st
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import models
import graph_rag as gr
import superb_client as sb

st.set_page_config(page_title="Wafer Defect Analyzer", layout="wide")

ALLOWED = {"png", "jpg", "jpeg", "bmp"}


# ---------------------------- 유틸 ----------------------------
def freq_surface(img: Image.Image):
    """F-03: 웨이퍼 맵의 2D FFT 크기 스펙트럼을 3D surface로."""
    m = models.image_to_mask(img, 52).astype(float)
    F = np.fft.fftshift(np.abs(np.fft.fft2(m)))
    F = np.log1p(F)
    fig = plt.figure(figsize=(5, 4))
    ax = fig.add_subplot(111, projection="3d")
    xx, yy = np.meshgrid(np.arange(F.shape[1]), np.arange(F.shape[0]))
    ax.plot_surface(xx, yy, F, cmap="viridis", linewidth=0, antialiased=True)
    ax.set_title("3D Frequency Spectrum (log|FFT|)", fontsize=9)
    ax.set_xlabel("u"); ax.set_ylabel("v"); ax.set_zlabel("mag")
    fig.tight_layout()
    return fig


def draw_boxes(img: Image.Image, boxes):
    """F-02: 원본 위에 빨간 박스."""
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img); ax.axis("off")
    import matplotlib.patches as patches
    for name, x0, y0, x1, y1, conf in boxes:
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                     fill=False, edgecolor="#ef4444", lw=2.2))
        ax.text(x0, max(y0 - 4, 0), f"{name} {conf:.2f}", color="white",
                fontsize=8, bbox=dict(facecolor="#ef4444", pad=1, edgecolor="none"))
    fig.tight_layout()
    return fig


def build_pdf(img, boxes, report_text, proc, preds, scores, src):
    """F-07: 리포트 + 시각화 종합 PDF (bytes)."""
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.units import mm
    from reportlab.pdfgen import canvas
    from reportlab.lib.utils import ImageReader

    buf = io.BytesIO()
    c = canvas.Canvas(buf, pagesize=A4)
    W, H = A4
    # 한글 폰트 (있으면 등록, 없으면 기본)
    font = "Helvetica"
    try:
        from reportlab.pdfbase import pdfmetrics
        from reportlab.pdfbase.ttfonts import TTFont
        for p in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
                  "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
            if os.path.exists(p):
                pdfmetrics.registerFont(TTFont("KR", p)); font = "KR"; break
    except Exception:
        pass

    c.setFont(font, 16); c.drawString(20 * mm, H - 25 * mm, "웨이퍼 결함 분석 리포트")
    c.setFont(font, 9)
    c.drawString(20 * mm, H - 32 * mm,
                 f"생성: {datetime.datetime.now():%Y-%m-%d %H:%M}   |   탐지 소스: {src}")

    # 이미지 (박스 포함)
    fig = draw_boxes(img, boxes)
    ib = io.BytesIO(); fig.savefig(ib, format="png", dpi=120, bbox_inches="tight"); plt.close(fig); ib.seek(0)
    c.drawImage(ImageReader(ib), 20 * mm, H - 120 * mm, width=80 * mm, height=80 * mm, preserveAspectRatio=True)

    # 3D 주파수
    fig2 = freq_surface(img)
    ib2 = io.BytesIO(); fig2.savefig(ib2, format="png", dpi=120, bbox_inches="tight"); plt.close(fig2); ib2.seek(0)
    c.drawImage(ImageReader(ib2), 110 * mm, H - 120 * mm, width=80 * mm, height=70 * mm, preserveAspectRatio=True)

    # 텍스트 리포트
    c.setFont(font, 11); c.drawString(20 * mm, H - 130 * mm, f"판단 공정: {proc}")
    c.setFont(font, 9)
    y = H - 138 * mm
    c.drawString(20 * mm, y, "예측 패턴: " + (", ".join(preds) if preds else "정상(결함 없음)")); y -= 6 * mm
    for line in report_text.split("\n"):
        for chunk in [line[i:i + 90] for i in range(0, len(line), 90)] or [""]:
            c.drawString(20 * mm, y, chunk); y -= 5 * mm
            if y < 20 * mm:
                c.showPage(); c.setFont(font, 9); y = H - 20 * mm
    c.showPage(); c.save(); buf.seek(0)
    return buf.getvalue()


# ---------------------------- 상태 ----------------------------
if "asset_id" not in st.session_state:
    st.session_state.asset_id = None
if "result" not in st.session_state:
    st.session_state.result = None

st.title("🔬 웨이퍼 결함 분석 · 공정 원인 역추적")
b_sb = "🟢 Superb 연동" if sb.available() else "⚪ Superb 오프라인"
b_dp = "🟢 배포모델(RF-DETR)" if sb.deploy_available() else "⚪ 배포모델 미설정"
b_yo = "🟢 YOLO best.pt" if models.yolo_available() else "⚪ YOLO 없음"
b_cl = "🟢 Swin 분류" if os.path.exists(models.CLS_PATH) else "⚪ Swin 없음"
st.caption(f"MixedWM38 · {b_sb} · {b_dp} · {b_yo} · {b_cl}")

left, center, right = st.columns([1.0, 1.4, 1.3], gap="large")

# ==================== 좌: 입력 & 공정 필터 (F-01, F-05, F-06) ====================
with left:
    st.subheader("1. 입력")
    proc_filter = st.selectbox("공정 카테고리 선택 필터", gr.PROCESS_CHOICES, index=0,
                               help="기본값: 자동 인식 (F-05)")
    up = st.file_uploader("웨이퍼 맵 이미지 (.png/.jpg)", type=list(ALLOWED),
                          help="드래그&드롭 지원 (F-01)")
    det_src = st.radio("탐지 엔진 (F-02)",
                       ["auto", "superb", "yolo", "heuristic"],
                       format_func=lambda s: {"auto": "자동 (Superb→YOLO→폴백)",
                                              "superb": "Superb 배포모델 (RF-DETR)",
                                              "yolo": "로컬 YOLOv8",
                                              "heuristic": "휴리스틱"}[s],
                       index=0, horizontal=False)
    thr = st.slider("분류 임계값", 0.2, 0.8, 0.5, 0.05)
    dconf = st.slider("탐지 신뢰도 임계값", 0.05, 0.9, 0.25, 0.05)
    run = st.button("🚀 분석 시작", use_container_width=True, type="primary")

    if up is not None:
        ext = up.name.rsplit(".", 1)[-1].lower()
        if ext not in ALLOWED:      # F-06 예외처리
            st.error("지원하지 않는 파일 형식입니다. 반도체 웨이퍼 맵 이미지 파일(.png, .jpg)을 등록해 주세요.")
            up = None

    if run and up is not None:
        try:
            img = Image.open(up).convert("RGB")
        except Exception:
            st.error("이미지를 열 수 없습니다. 손상되지 않은 웨이퍼 맵 파일인지 확인해 주세요.")
            img = None
        if img is not None:
            # F-01: Superb 저장소 업로드 (연동 시)
            tmp = f"/tmp/{up.name}"; img.save(tmp)
            upres = sb.upload_image(tmp, key=up.name)
            st.session_state.asset_id = upres.get("asset_id")
            # 추론
            preds, scores, csrc = models.classify(img, thr=thr)
            boxes, dsrc = models.detect(img, source=det_src, conf=dconf)
            conf = max([scores[p] for p in preds], default=None)
            st.session_state.result = dict(img=img, preds=preds, scores=scores, csrc=csrc,
                                           boxes=boxes, dsrc=dsrc, conf=conf)
            # 오토라벨링 결과 Superb 적재
            if st.session_state.asset_id and preds:
                sb.push_prediction(st.session_state.asset_id, preds)
            st.success(f"분석 완료 · 분류:{csrc} · 탐지:{dsrc}"
                       + (f" · Superb 적재 asset={st.session_state.asset_id}" if st.session_state.asset_id else ""))
    elif run and up is None:
        st.warning("먼저 이미지를 업로드하세요.")

# ==================== 중앙: 탐지 + 3D 주파수 (F-02, F-03) ====================
with center:
    st.subheader("2. 결함 탐지 & 주파수 분석")
    R = st.session_state.result
    if R:
        st.pyplot(draw_boxes(R["img"], R["boxes"]), use_container_width=True)
        st.caption(f"F-02 탐지 소스: {R['dsrc']} · 박스 {len(R['boxes'])}개")
        st.pyplot(freq_surface(R["img"]), use_container_width=True)
        st.caption("F-03 3D 주파수 스펙트럼 — 규칙적 패턴일수록 특정 주파수에 에너지 집중")
    else:
        st.info("좌측에서 이미지를 업로드하고 [분석 시작]을 누르세요.")

# ==================== 우: 리포트 + 피드백 + PDF (F-04, F-05, F-07) ====================
with right:
    st.subheader("3. 원인 역추적 리포트")
    R = st.session_state.result
    if R:
        preds = R["preds"]
        use_llm = st.toggle("로컬 LLM 문장화(GPU 필요)", value=False)
        proc = gr.primary_process(preds) if preds else "정상"
        report_text = gr.report(preds, confidence=R["conf"], use_llm=use_llm)
        st.markdown(f"**판단 공정:** {proc}")
        if R["conf"] is not None:
            st.progress(min(R["conf"], 1.0), text=f"신뢰도 {R['conf']:.0%}")
        st.text_area("리포트 (F-04)", report_text, height=180)

        # 예측 점수 표
        st.caption("패턴별 점수")
        st.bar_chart({k: v for k, v in R["scores"].items()})

        # F-05: 공정 강제 수정 + 피드백
        st.divider()
        st.markdown("**F-05 · AI 판단 강제 수정 (액티브 러닝)**")
        corrected = st.multiselect("실제 결함 패턴으로 수정", models.LABELS, default=preds)
        cols = st.columns(2)
        if cols[0].button("피드백 전송", use_container_width=True):
            if st.session_state.asset_id:
                fr = sb.push_feedback(st.session_state.asset_id, corrected)
                st.success("Superb로 피드백 전송됨" if fr.get("ok") else f"전송 실패: {fr.get('reason')}")
            else:
                st.info("Superb 오프라인 — 수정값은 로컬 리포트에만 반영됩니다.")
            R["preds"] = corrected
            R["conf"] = max([R["scores"].get(p, 0.6) for p in corrected], default=None)

        # F-07: PDF 다운로드
        st.divider()
        final_preds = R["preds"]
        pdf = build_pdf(R["img"], R["boxes"],
                        gr.report(final_preds, confidence=R["conf"], use_llm=False),
                        gr.primary_process(final_preds) if final_preds else "정상",
                        final_preds, R["scores"], R["dsrc"])
        cols[1].download_button("📄 PDF 리포트", pdf,
                                file_name="wafer_report.pdf", mime="application/pdf",
                                use_container_width=True)
    else:
        st.info("분석 결과가 여기에 표시됩니다.")


In [ ]:
# 학습된 가중치를 앱 폴더로 (있으면)
import shutil, glob, os
best=glob.glob('runs/detect/wafer_det*/weights/best.pt')
if best: shutil.copy(best[0],'best.pt'); print('best.pt 복사')
if os.path.exists('swin_multilabel.pt'): print('swin_multilabel.pt 준비됨')

### 5-1. Colab에서 Streamlit 실행 (localtunnel)
아래 실행 후 나오는 **URL** 클릭 → 비밀번호 칸엔 그 아래 출력된 **IP**를 입력하세요.

In [ ]:
!npm install -g localtunnel >/dev/null 2>&1
!streamlit run app.py &>/content/st_log.txt &
import time; time.sleep(6)
print('접속 비밀번호(IP):')
!wget -q -O - https://loca.lt/mytunnelpassword; print()
!npx --yes localtunnel --port 8501

## 논문 매핑
- 2: 단일패턴 탐지(YOLO) — F-02 데모
- 3: 다중라벨 분류(Swin, 38k) — **다중 결합 패턴 인식 = 논문 핵심**, macro-F1/혼동행렬
- 4: Graph-RAG 원인 역추적 — 설명가능성 기여, groundedness 지표
- 5: Streamlit 통합 데모 — F-01~F-07

**역할 분담**: Superb=단일패턴 탐지 + 라벨링/오토라벨링 플랫폼(F-01/F-05), 노트북=다중패턴 분류(핵심).